# Stress Level Classifier — Scraped Data Collection Notebook

Use this notebook inside VS Code with your `ds` Anaconda environment selected as the kernel.

Goal: collect an additional scraped dataset from Stack Exchange API and save it as `scraped_stackexchange_stress_v2.csv`.

The output labels are initial weak labels, not final labels. The rows should be manually reviewed before final modeling.

## 1. Imports and settings

In [1]:
import csv
import html
import json
import random
import re
import time
from datetime import date
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd

OUTPUT_FILE = "../data/raw/scraped/scraped_stackexchange_stress_v2.csv"

# To get around 1000 total rows, aim for 350 per class.
# The API may return fewer rows for some classes, especially High.
TARGET_PER_CLASS = 350

# Increase this if you still need more rows.
PAGES_PER_QUERY = 5

# Small delay to avoid hitting the API too aggressively.
DELAY_SECONDS = 0.35

MIN_WORDS = 20
MAX_WORDS = 500

STANDARD_COLUMNS = [
    "id", "text", "stress_level", "source_type", "source_name", "source_url",
    "source_record_id", "original_label", "label_mapping_rule", "language",
    "date_collected", "collector", "labeler", "review_status", "notes"
]

Path("../data/raw/scraped").mkdir(parents=True, exist_ok=True)
Path("../data/interim").mkdir(parents=True, exist_ok=True)


## 2. Expanded query plan

These queries are intentionally broad because scraped data is expected to be noisy. We will review/clean it later.

In [2]:
QUERY_GROUPS = {
    "High": [
        "overwhelmed",
        "burnout",
        "panic attack",
        "anxiety stress",
        "extreme stress",
        "cannot cope",
        "mental breakdown",
        "work stress burnout",
        "deadline anxiety",
        "exam stress overwhelmed",
        "exhausted pressure",
        "too much pressure",
        "constant anxiety",
        "severe stress",
        "stressed out",
    ],
    "Medium": [
        "worried deadline",
        "nervous presentation",
        "workload stress",
        "concerned grades",
        "busy schedule",
        "time management problem",
        "exam preparation worry",
        "project deadline concern",
        "mild anxiety",
        "pressure at work",
        "difficult semester",
        "job interview nervous",
        "uncertain about decision",
        "need advice stress",
        "handling pressure",
    ],
    "Low": [
        "calm routine",
        "relaxed schedule",
        "balanced workload",
        "productive day",
        "good time management",
        "healthy routine",
        "study schedule working",
        "work life balance",
        "feeling confident",
        "comfortable with workload",
        "manageable workload",
        "normal routine",
        "organized schedule",
        "positive habits",
        "not stressed",
    ],
}

# Some site names may return fewer results. Warnings are okay.
SITES = [
    "academia",
    "workplace",
    "interpersonal",
    "parenting",
    "pm",       # Project Management
    "money",    # Personal Finance
    "fitness",
    "health",
]


## 3. Helper functions

In [3]:
def strip_html(value: str) -> str:
    value = html.unescape(value or "")
    value = re.sub(r"<script.*?</script>", " ", value, flags=re.I | re.S)
    value = re.sub(r"<style.*?</style>", " ", value, flags=re.I | re.S)
    value = re.sub(r"<[^>]+>", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

def stackexchange_search(site: str, query: str, page: int, pagesize: int = 50):
    params = {
        "order": "desc",
        "sort": "activity",
        "q": query,
        "site": site,
        "filter": "withbody",
        "page": page,
        "pagesize": pagesize,
    }
    url = "https://api.stackexchange.com/2.3/search/advanced?" + urlencode(params)
    req = Request(url, headers={"User-Agent": "stress-level-classifier-course-project/1.0"})
    with urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode("utf-8"))

def build_text(item: dict) -> str:
    title = strip_html(item.get("title", ""))
    body = strip_html(item.get("body", ""))
    text = f"{title}. {body}".strip()
    text = re.sub(r"\s+", " ", text)
    words = text.split()
    if len(words) > MAX_WORDS:
        text = " ".join(words[:MAX_WORDS])
    return text

def is_usable_text(text: str) -> bool:
    if not isinstance(text, str):
        return False
    words = text.split()
    if len(words) < MIN_WORDS:
        return False
    # Exclude extremely sensitive content from scraped component.
    blocked = ["suicide", "kill myself", "self harm", "self-harm"]
    lowered = text.lower()
    return not any(term in lowered for term in blocked)


## 4. Run scraping collection

This may take a few minutes. If the API gives warnings for some sites, continue; the notebook skips failed queries.

In [4]:
def collect_scraped_rows(target_per_class=TARGET_PER_CLASS):
    rows = []
    seen_texts = set()
    today = date.today().isoformat()

    for stress_level, queries in QUERY_GROUPS.items():
        class_rows = []
        print(f"\nCollecting class: {stress_level}")

        for site in SITES:
            for query in queries:
                for page in range(1, PAGES_PER_QUERY + 1):
                    if len(class_rows) >= target_per_class:
                        break
                    try:
                        data = stackexchange_search(site, query, page=page, pagesize=50)
                    except Exception as exc:
                        print(f"  Warning: site={site}, q={query!r}, page={page}: {exc}")
                        continue

                    for item in data.get("items", []):
                        if len(class_rows) >= target_per_class:
                            break

                        text = build_text(item)
                        if not is_usable_text(text):
                            continue

                        key = text.lower().strip()
                        if key in seen_texts:
                            continue
                        seen_texts.add(key)

                        class_rows.append({
                            "id": "",
                            "text": text,
                            "stress_level": stress_level,
                            "source_type": "online_scraping_api",
                            "source_name": "Stack Exchange API",
                            "source_url": item.get("link", ""),
                            "source_record_id": str(item.get("question_id", "")),
                            "original_label": f"{site}:{query}",
                            "label_mapping_rule": "expanded_keyword_query_mapping_v2",
                            "language": "English",
                            "date_collected": today,
                            "collector": "project_team",
                            "labeler": "keyword_mapping_needs_manual_review",
                            "review_status": "needs_review",
                            "notes": "Scraped online text; verify label manually before final modeling.",
                        })

                    if not data.get("has_more", False):
                        break
                    time.sleep(DELAY_SECONDS)

                if len(class_rows) >= target_per_class:
                    break
            if len(class_rows) >= target_per_class:
                break

        print(f"  collected {len(class_rows)} rows for {stress_level}")
        rows.extend(class_rows)

    random.seed(42)
    random.shuffle(rows)
    for index, row in enumerate(rows, start=1):
        row["id"] = f"scraped_{index:05d}"
    return rows

rows = collect_scraped_rows(TARGET_PER_CLASS)
scraped_df = pd.DataFrame(rows, columns=STANDARD_COLUMNS)
scraped_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

print(f"\nSaved {len(scraped_df)} rows to {OUTPUT_FILE}")
print(scraped_df["stress_level"].value_counts())



  collected 350 rows for High

  collected 350 rows for Medium

  collected 350 rows for Low

Saved 1050 rows to scraped_stackexchange_stress_v2.csv
stress_level
High      350
Low       350
Medium    350
Name: count, dtype: int64


## 5. Inspect the scraped data

In [5]:
scraped_df = pd.read_csv(OUTPUT_FILE)

print("Shape:", scraped_df.shape)
print("\nClass counts:")
print(scraped_df["stress_level"].value_counts())
print("\nMissing text:", scraped_df["text"].isna().sum())
print("Duplicate text:", scraped_df["text"].duplicated().sum())

scraped_df[["text", "stress_level", "original_label", "source_url"]].sample(
    min(10, len(scraped_df)), random_state=42
)


Shape: (1050, 15)

Class counts:
stress_level
High      350
Low       350
Medium    350
Name: count, dtype: int64

Missing text: 0
Duplicate text: 0


,text,stress_level,original_label,source_url
352,Changing fields during PhD. I’m three months i...,High,academia:overwhelmed,https://academia.stackexchange.com/questions/2...
689,Are unimpressive grades in core courses an ind...,Medium,academia:concerned grades,https://academia.stackexchange.com/questions/1...
485,"Two-author paper with thesis mentor, but the i...",Low,academia:not stressed,https://academia.stackexchange.com/questions/2...
388,Regarding hiding depression during application...,Low,academia:normal routine,https://academia.stackexchange.com/questions/1...
31,How to advise a researcher facing a choice bet...,Low,academia:not stressed,https://academia.stackexchange.com/questions/2...
442,Failed my BEng dissertation in a computer/elec...,High,academia:panic attack,https://academia.stackexchange.com/questions/1...
198,How can I overcome the setbacks of a negligent...,Low,academia:good time management,https://academia.stackexchange.com/questions/1...
425,Letter writer snapped at my reminder email. Wh...,Medium,academia:worried deadline,https://academia.stackexchange.com/questions/1...
107,What should I put in the introduction chapter ...,Low,academia:work life balance,https://academia.stackexchange.com/questions/1...
714,Teaching a very small class when the students ...,Low,academia:not stressed,https://academia.stackexchange.com/questions/2...


## 6. Optional: combine with previous scraped file

Run this only if you want to combine the old `scraped_stackexchange_stress.csv` file with this new v2 file.

In [6]:
old_file = Path("../data/raw/scraped/scraped_stackexchange_stress.csv")
new_file = Path("../data/raw/scraped/scraped_stackexchange_stress_v2.csv")
combined_scraped_file = Path("../data/interim/scraped_stackexchange_stress_combined.csv")

if old_file.exists() and new_file.exists():
    old_df = pd.read_csv(old_file)
    new_df = pd.read_csv(new_file)
    combined_scraped = pd.concat([old_df, new_df], ignore_index=True)
    combined_scraped["text_norm"] = combined_scraped["text"].astype(str).str.lower().str.strip()
    combined_scraped = combined_scraped.drop_duplicates(subset=["text_norm"]).drop(columns=["text_norm"])
    combined_scraped["id"] = [f"scraped_combined_{i:05d}" for i in range(1, len(combined_scraped) + 1)]
    combined_scraped.to_csv(combined_scraped_file, index=False, encoding="utf-8")
    print(f"Saved {len(combined_scraped)} rows to {combined_scraped_file}")
    print(combined_scraped["stress_level"].value_counts())
else:
    print("Old or new scraped file not found. Skipping combine step.")


Saved 1240 rows to scraped_stackexchange_stress_combined.csv
stress_level
Low       493
Medium    393
High      354
Name: count, dtype: int64
